In [0]:
df = spark.table("workspace.default.daily_delhi_climate_train")
df.orderBy("date")



In [0]:
total_rows = df.count()
batches = 5
batch_size = total_rows / batches

from pyspark.sql.functions import monotonically_increasing_id

# Adding an index row to help in making the batches
df_indexed = df.withColumn("row_id", monotonically_increasing_id())

# Creting 5 batches from the training data
for i in range(batches):
    start = i * batch_size
    end = (i + 1) * batch_size if i < 4 else total_rows

    batch_df = df_indexed.filter((df_indexed.row_id >= start) & (df_indexed.row_id < end))

    batch_df.write.format("delta").mode("overwrite").saveAsTable(f"workspace.default.train_batch_{i+1}")

